Assigment 3
```
# COSC5437002 - Neural Networks and Deep Learning
# Prof. Syed Muhammad Danish
# Algoma University
# Department of Computer Science and Mathematics
# Brampton, Ontario, Canada
# Date: 8 jul 2025
```

# Final CNN:
- Added Data Augmentation
- Added ResNet18 + SGD
- Added scheduler
- Added Early Stopping



In [ ]:
import matplotlib.pyplot as plt
import torchvision
import numpy as np
from torchvision import transforms
import torch
from torchvision import datasets
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.optim as optim
import torch.nn.init as init
# Import ResNet
from torchvision.models import resnet18

In [ ]:
transform = transforms.Compose([ # Transform: flatten the image to a vector
    transforms.RandomHorizontalFlip(), # ADDED
    transforms.RandomCrop(32, padding=4), # ADDED
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),  # normalize to [-1, 1]
])

train_dataset = datasets.CIFAR100(root='./data', train=True, download=True, transform=transform)
print(f"Total training samples: {len(train_dataset)}")
test_dataset = datasets.CIFAR100(root='./data', train=False, download=True, transform=transform)
print(f"Total test samples: {len(test_dataset)}")

# ~> Setting batch size as 32
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

Total training samples: 50000
Total test samples: 10000
cuda


In [ ]:
# Using a pre-trained ResNet-18 model and modifying the final layer for CIFAR-100
model = resnet18(weights=None) # weights=None means random initialization
# Modify the first convolutional layer for CIFAR-100 images (3 channels, 32x32)
# ResNet's default conv1 is 7x7 stride 2, which is too aggressive for 32x32 images.
# We change it to 3x3 stride 1, no pooling.
model.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
model.maxpool = nn.Identity()
# Modify the final fully connected layer for 100 classes (CIFAR-100)
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 100) # CIFAR-100 has 100 classes

model = model.to(device)

criterion = nn.CrossEntropyLoss()
# ResNet models often benefit from a smaller initial learning rate.
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=5e-4)

from torch.optim.lr_scheduler import StepLR
scheduler = StepLR(optimizer, step_size=7, gamma=0.1)

# Early Stopping Parameters
patience = 5 # How many epochs to wait after last time validation accuracy improved.
min_delta = 0.001 # Minimum change in the monitored quantity to qualify as an improvement.

best_accuracy = -1.0
epochs_no_improve = 0

In [ ]:
# Training loop with weight visualization
numEpoch = 20
for epoch in range(numEpoch):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    scheduler.step()

    train_accuracy = correct/total*100
    print(f"\nEpoch [{epoch+1}/{numEpoch}] Loss: {total_loss:.4f} Training Accuracy: {train_accuracy:.2f}%")

    model.eval()
    test_correct = 0
    test_total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            test_correct += (predicted == labels).sum().item()
            test_total += labels.size(0)

    current_test_accuracy = (test_correct / test_total) * 100
    print(f"Current Test Accuracy: {current_test_accuracy:.2f}%")

    if current_test_accuracy > best_accuracy + min_delta:
        best_accuracy = current_test_accuracy
        epochs_no_improve = 0
        torch.save(model.state_dict(), 'best_model.pth')
        print(f"New best test accuracy: {best_accuracy:.2f}%. Model saved.")
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs.")
        if epochs_no_improve >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs. Best Test Accuracy: {best_accuracy:.2f}%")
            break

print(f"\nFinal Test Accuracy after training (or early stopping): {best_accuracy:.2f}%")


Epoch [1/20] Loss: 5935.7680 Training Accuracy: 11.59%
Current Test Accuracy: 16.48%
New best test accuracy: 16.48%. Model saved.

Epoch [2/20] Loss: 4737.4430 Training Accuracy: 24.25%
Current Test Accuracy: 29.53%
New best test accuracy: 29.53%. Model saved.

Epoch [3/20] Loss: 3905.7791 Training Accuracy: 34.73%
Current Test Accuracy: 39.11%
New best test accuracy: 39.11%. Model saved.

Epoch [4/20] Loss: 3332.7808 Training Accuracy: 42.57%
Current Test Accuracy: 45.87%
New best test accuracy: 45.87%. Model saved.

Epoch [5/20] Loss: 2937.9320 Training Accuracy: 48.33%
Current Test Accuracy: 47.25%
New best test accuracy: 47.25%. Model saved.

Epoch [6/20] Loss: 2668.7508 Training Accuracy: 52.15%
Current Test Accuracy: 52.86%
New best test accuracy: 52.86%. Model saved.

Epoch [7/20] Loss: 2452.7784 Training Accuracy: 55.75%
Current Test Accuracy: 54.93%
New best test accuracy: 54.93%. Model saved.

Epoch [8/20] Loss: 1881.0869 Training Accuracy: 65.74%
Current Test Accuracy: 63.1